# Traces en vrac pour check cas limites affiliation

In [ ]:
# ========= Diagnostic des cas concernés par le fallback ==========

# Cas concernés : affiliation mandat manquante, mais groupeAbrev disponible
mask_fallback = df["affiliation_mandat_députés"].isna() & df["groupeAbrev"].notna()

fallback_cases = df.loc[
    mask_fallback,
    [
        "id_acteur",
        "nom_orateur_clean",
        "qualite_orateur",
        "groupeAbrev",
        "dateSeance_ts",
    ],
].copy()

# Reconstituer l'affiliation qui serait attribuée par le fallback
fallback_cases["affiliation_fallback"] = fallback_cases["groupeAbrev"]
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    recodage_affiliation
)
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    {"LES-REP": "LR", "UMP": "LR"}
)

# Print des infos
print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", len(fallback_cases))
print("Nombre d'id_acteur uniques :", fallback_cases["id_acteur"].nunique(dropna=True))
print(
    "Nombre d'orateurs uniques :",
    fallback_cases["nom_orateur_clean"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(fallback_cases["nom_orateur_clean"].dropna().unique())

display(
    fallback_cases[
        [
            "id_acteur",
            "nom_orateur_clean",
            "groupeAbrev",
            "affiliation_fallback",
        ]
    ]
    .value_counts()
    .reset_index(name="n_interventions")
)

In [ ]:
# ========= Cas limites GOUV =============

# DANS CAS AFFILIATION DYNAMIQUE DÉPUTÉS QUI SONT GOUV

# vérification des cas sans affiliation :
print(
    "Nombre d'id_acteur uniques sans affiliation :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)
print("\nValeur counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation_mandat_députés"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)

# Vérifier les cas où affiliation n'est pas nulle mais avec qualité orateur spécifique
# = membres du gouv mais qui sont députés et flaguent donc avec une affiliation députés

mask_affil_with_qualite = df["affiliation_mandat_députés"].notna() & df[
    "qualite_orateur"
].str.contains("ministre|garde des sceaux|secrétaire d'État", case=False, na=False)

print(
    "Nombre de lignes avec affiliation ET qualité gouvernementale :",
    mask_affil_with_qualite.sum(),
)
print("\nAffiliations pour ces cas :")
print(
    df.loc[mask_affil_with_qualite, "affiliation_mandat_députés"].value_counts().head()
)

print("\nExemples de ces lignes :")
print(
    df.loc[
        mask_affil_with_qualite,
        ["nom_orateur_clean", "qualite_orateur", "affiliation_mandat_députés"],
    ]
    .drop_duplicates()
    .head()
)


## Pour dif id acteur vs id orateur :
Pas un pb de notre code :

parfois bug de leur fichier : deux bloc orateurs qui s'enchainent
exemple : id_syceron 3180585 dans CRSANR5L16S2023O1N292

        <paragraphe valeur_ptsodj="3" ordinal_prise="8" id_preparation="2318149" ordre_absolu_seance="348" id_acteur="PA1567" id_mandat="PM797631" id_nomination_oe="-1" id_nomination_op="-1" code_grammaire="INTERRUPTION_1_10" code_style="NORMAL" code_parole="" sommaire="0" id_syceron="3180585" valeur="795636;0 1567;0">
          <orateurs>
            <orateur>
              <nom>M. Benjamin Lucas</nom>
              <id>795636</id>
              <qualite/>
            </orateur>
            <orateur>
              <nom>M. Jérôme Guedj</nom>
              <id>1567</id>
              <qualite/>
            </orateur>
          </orateurs>
          <texte stime="7965.54">Il fallait voter l’augmentation du Smic !</texte>

Et parfois erreur de code id juste , genre ici :
rudigoz cause
tavel gueule en interruption
rudigoz continue
la président dit a tavel de pose sons cul sur sa chaise
clouet fini par brailler -> ils passent l'id_Acteur de tavel

        <paragraphe valeur_ptsodj="2" ordinal_prise="6" id_preparation="2207403" ordre_absolu_seance="310" id_acteur="PA722292" id_mandat="PM797241" id_nomination_oe="-1" id_nomination_op="-1" code_grammaire="PAROLE_GENERIQUE" code_style="NORMAL" code_parole="PAROLE_1_2" sommaire="0" id_syceron="3024322" type_debat="PLFSS" valeur="">
          <orateurs>
            <orateur>
              <nom>M. Thomas Rudigoz</nom>
              <id>722292</id>
              <qualite/>
            </orateur>
          </orateurs>
          <texte stime="5409.51">…comme lorsque M. Tavel attaque, encore une fois, M. le ministre du travail, du plein emploi et de l’insertion. Après les différents dérapages de votre groupe, vous poursuivez dans l’outrance et la violence verbales.</texte>
        </paragraphe>
        <paragraphe valeur_ptsodj="2" ordinal_prise="6" id_preparation="2207405" ordre_absolu_seance="311" id_acteur="PA794166" id_mandat="PM796749" id_nomination_oe="-1" id_nomination_op="-1" code_grammaire="INTERRUPTION_1_10" code_style="NORMAL" code_parole="" sommaire="0" id_syceron="3024324" type_debat="PLFSS" valeur="793736;0">
          <orateurs>
            <orateur>
              <nom>M. Hadrien Clouet</nom>
              <id>793736</id>
              <qualite/>
            </orateur>
          </orateurs>
          <texte stime="5420.64">Vous n’avez qu’à nous répondre !</texte>
        </paragraphe>